In [7]:
import os
import glob
import pandas as pd

anios_validos = [str(anio) for anio in range(2016, 2027)]

# 1. LOAD RETAIL SALES
archivos_ventas = glob.glob("../data/retail_sales/**/*.csv", recursive=True)
archivos_ventas_10y = [f for f in archivos_ventas if any(a in os.path.basename(f) for a in anios_validos)]

df_ventas = pd.concat((pd.read_csv(f, encoding='utf-8') for f in archivos_ventas_10y), ignore_index=True)

print(f"Retail Sales loaded: {len(df_ventas):,} rows ({len(archivos_ventas_10y)} files)")
print("Sales Columns:", df_ventas.columns.tolist())
print("Unique Brands in Sales:", sorted(df_ventas['MARCA'].unique()))
print("-" * 50)

# 2. LOAD EV & HYBRIDS
archivos_hyb = glob.glob("../data/ev_hybrids/**/*.csv", recursive=True)
archivos_hyb_10y = [f for f in archivos_hyb if any(a in os.path.basename(f) for a in anios_validos)]

df_hyb = pd.concat((pd.read_csv(f, encoding='utf-8') for f in archivos_hyb_10y), ignore_index=True)

print(f"EV/Hybrids loaded: {len(df_hyb):,} rows ({len(archivos_hyb_10y)} files)")
print("EV/Hybrids Columns:", df_hyb.columns.tolist())
display(df_hyb.head(3))

Retail Sales loaded: 60,551 rows (11 files)
Sales Columns: ['PROD_EST', 'COBERTURA', 'ANIO', 'ID_MES', 'MARCA', 'MODELO', 'TIPO', 'SEGMENTO', 'ORIGEN', 'ID_PAIS_ORIGEN', 'UNI_VEH', 'ESTATUS']
Unique Brands in Sales: ['Acura', 'Alfa Romeo', 'Audi', 'Auteco', 'BMW', 'Bentley', 'Changan', 'Chirey', 'Chrysler', 'Fiat', 'Ford Motor', 'Foton', 'GAC', 'Geely', 'General Motors', 'Great Wall Motor', 'Honda', 'Hyundai', 'Infiniti', 'Isuzu', 'JAC', 'JETOUR', 'Jaguar', 'Jetour Soueast', 'KIA', 'Land Rover', 'Leapmotor', 'Lexus', 'Lincoln', 'Lynk&Co', 'MG Motor', 'MOTORNATION', 'Mazda', 'Mercedes Benz', 'Mini', 'Mitsubishi', 'Nissan', 'Omoda', 'Peugeot', 'Porsche', 'Renault', 'SEAT', 'Smart', 'Subaru', 'Suzuki', 'Toyota', 'Volkswagen', 'Volvo', 'Zeekr']
--------------------------------------------------
EV/Hybrids loaded: 4,252 rows (11 files)
EV/Hybrids Columns: ['PROD_EST', 'COBERTURA', 'ANIO', 'ID_MES', 'ID_ENTIDAD', 'VEH_ELECTR', 'VEH_HIBRIDAS_PLUGIN', 'VEH_HIBRIDAS', 'ESTATUS']


,PROD_EST,COBERTURA,ANIO,ID_MES,ID_ENTIDAD,VEH_ELECTR,VEH_HIBRIDAS_PLUGIN,VEH_HIBRIDAS,ESTATUS
0,Registro Administrativo de la Industria Automo...,Nacional,2016,1,1,0,0,2,Cifras Definitivas
1,Registro Administrativo de la Industria Automo...,Nacional,2016,1,2,0,0,1,Cifras Definitivas
2,Registro Administrativo de la Industria Automo...,Nacional,2016,1,3,0,0,1,Cifras Definitivas


In [ ]:
import os
import glob
import pandas as pd

# Create folder for processed data
os.makedirs("../data/processed", exist_ok=True)
valid_years = [str(year) for year in range(2016, 2027)]

# 1. CLEAN RETAIL SALES DATA
sales_files = glob.glob("../data/retail_sales/**/*.csv", recursive=True)
sales_files_10y = [f for f in sales_files if any(y in os.path.basename(f) for y in valid_years)]
df_sales = pd.concat((pd.read_csv(f, encoding='utf-8') for f in sales_files_10y), ignore_index=True)

# Standard monthly date
df_sales['FECHA'] = pd.to_datetime(
    df_sales['ANIO'].astype(str) + '-' + df_sales['ID_MES'].astype(str).str.zfill(2) + '-01'
)

# Harmonize brands into corporate groups (e.g., Stellantis, Jetour, Chery)
group_map = {
    'Chrysler': 'Stellantis',
    'Fiat': 'Stellantis',
    'Alfa Romeo': 'Stellantis',
    'Peugeot': 'Stellantis',
    'Leapmotor': 'Stellantis',
    'JETOUR': 'Jetour / Soueast',
    'Jetour Soueast': 'Jetour / Soueast',
    'Chirey': 'Chery Group (Chirey/Omoda)',
    'Omoda': 'Chery Group (Chirey/Omoda)'
}
df_sales['GRUPO_AUTOMOTRIZ'] = df_sales['MARCA'].replace(group_map)

# Classify corporate origin (Chinese brands reporting in RAIAVL vs. Traditional/Others)
chinese_brands = [
    'MG Motor', 'Chirey', 'Omoda', 'JAC', 'Changan', 'GAC', 
    'Great Wall Motor', 'Geely', 'Foton', 'Zeekr', 'Lynk&Co', 
    'JETOUR', 'Jetour Soueast', 'MOTORNATION', 'Leapmotor'
]
df_sales['ORIGEN_MARCA'] = df_sales['MARCA'].apply(
    lambda x: 'Chinese Brands (in RAIAVL)' if x in chinese_brands else 'Traditional / Other Brands'
)

# Translate status labels for English charts
status_map = {
    'Cifras Definitivas': 'Revised Figures',
    'Cifras Preliminares': 'Preliminary Figures'
}
df_sales['ESTATUS_EN'] = df_sales['ESTATUS'].replace(status_map)

df_sales.to_csv("../data/processed/clean_retail_sales.csv", index=False, encoding='utf-8')
print("Clean retail sales saved:", df_sales.shape)

# 2. CLEAN EV & HYBRIDS DATA
hyb_files = glob.glob("../data/ev_hybrids/**/*.csv", recursive=True)
hyb_files_10y = [f for f in hyb_files if any(y in os.path.basename(f) for y in valid_years)]
df_hyb = pd.concat((pd.read_csv(f, encoding='utf-8') for f in hyb_files_10y), ignore_index=True)

df_hyb['FECHA'] = pd.to_datetime(
    df_hyb['ANIO'].astype(str) + '-' + df_hyb['ID_MES'].astype(str).str.zfill(2) + '-01'
)

# Official INEGI state codes
inegi_states = {
    1: 'Aguascalientes', 2: 'Baja California', 3: 'Baja California Sur', 4: 'Campeche',
    5: 'Coahuila', 6: 'Colima', 7: 'Chiapas', 8: 'Chihuahua', 9: 'Mexico City',
    10: 'Durango', 11: 'Guanajuato', 12: 'Guerrero', 13: 'Hidalgo', 14: 'Jalisco',
    15: 'State of Mexico', 16: 'Michoacan', 17: 'Morelos', 18: 'Nayarit',
    19: 'Nuevo Leon', 20: 'Oaxaca', 21: 'Puebla', 22: 'Queretaro', 23: 'Quintana Roo',
    24: 'San Luis Potosi', 25: 'Sinaloa', 26: 'Sonora', 27: 'Tabasco', 28: 'Tamaulipas',
    29: 'Tlaxcala', 30: 'Veracruz', 31: 'Yucatan', 32: 'Zacatecas', 99: 'Unspecified'
}
df_hyb['ESTADO'] = df_hyb['ID_ENTIDAD'].map(inegi_states).fillna('Other / Unspecified')

df_hyb['TOTAL_ELECTRIFICADOS'] = (
    df_hyb['VEH_ELECTR'] + df_hyb['VEH_HIBRIDAS_PLUGIN'] + df_hyb['VEH_HIBRIDAS']
)

df_hyb.to_csv("../data/processed/clean_ev_hybrids.csv", index=False, encoding='utf-8')
print("Clean EV/Hybrids saved:", df_hyb.shape)

Clean retail sales saved: (60551, 16)
Clean EV/Hybrids saved: (4252, 12)
